# CardioVis - Rerun Missing Cases Only

Notebook phụ để rerun riêng các case `[MISSING]` bằng manual overrides.

- Không chạy lại toàn bộ pipeline.
- Chỉ rerun các dòng đã bị `source_not_found` trong lần chạy trước.
- Dùng các path override bạn đã chốt cho Patient 7 và Patient 9.
- Chỉ tạo video stage cho missed cases: `final/*_missed_only.mp4`.
- Extract frames từ chính các video `*_missed_only.mp4`.


In [ ]:
from google.colab import drive
from pathlib import Path
import csv

drive.mount('/content/drive')

# Script path (điều chỉnh nếu bạn đặt script ở nơi khác)
SCRIPT_PATH = Path('/content/drive/MyDrive/Cardiovis-related/colab_drive_stage_pipeline.py')
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f'Pipeline script not found: {SCRIPT_PATH}')

# Tạo manual overrides CSV cho các case bạn đã chọn
override_rows = [
    {
        'patient': 7,
        'video_index': 2,
        'file_path': '/content/drive/MyDrive/3) Videos/Valve replacement/Patient 7/2026-01-28_141117 Pham Văn Toán N26-0024090 done/2026-01-28_141117_VID002.mp4',
    },
    {
        'patient': 7,
        'video_index': 1,
        'file_path': '/content/drive/MyDrive/3) Videos/Valve replacement/Patient 7/2026-01-28_141117 Pham Văn Toán N26-0024090 done/2026-01-28_141117_VID001.mp4',
    },
    {
        'patient': 9,
        'video_index': 3,
        'file_path': '/content/drive/MyDrive/3) Videos/Valve Repair/Patient 9/Full video/2025-04-02_093426_VID003.mp4',
    },
]

overrides_csv = Path('/content/drive/MyDrive/Cardiovis-related/manual_goc_overrides.csv')
overrides_csv.parent.mkdir(parents=True, exist_ok=True)
with overrides_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['patient', 'video_index', 'file_path'])
    writer.writeheader()
    writer.writerows(override_rows)
print(f'Wrote overrides CSV: {overrides_csv}')


In [ ]:
import os

# Paths
os.environ['DOCX_PATH'] = '/content/drive/MyDrive/passio-lab-1.docx'
os.environ['MERGED_DIR'] = '/content/drive/MyDrive/CardioVis-merged-videos'
os.environ['ROOT_VIDEOS_DIR'] = '/content/drive/MyDrive/3) Videos'
os.environ['CARDIOVIS_RELATED_DIR'] = '/content/drive/MyDrive/Cardiovis-related'
os.environ['AUTODETECT_DRIVE_PATHS'] = '1'

# Rerun only missing-source cases from previous report
os.environ['RERUN_SOURCE_NOT_FOUND_ONLY'] = '1'
os.environ['MANUAL_GOC_OVERRIDES_CSV'] = '/content/drive/MyDrive/Cardiovis-related/manual_goc_overrides.csv'
os.environ['STRICT_AMBIGUOUS_GOC'] = '1'
os.environ['RERUN_AFFECTED_STAGES_ONLY'] = '1'   # only merge stages touched by rerun
os.environ['MISSED_ONLY_STAGE_OUTPUTS'] = '1'     # create final/*_missed_only.mp4 only from rerun cases

# Keep frames extraction for missed-only outputs
os.environ['SKIP_FRAME_EXTRACT'] = '0'
os.environ['SKIP_SAMPLE_DOWNLOAD'] = '1'

# Keep main output behavior
os.environ['DRY_RUN'] = '0'
os.environ['EXTRACT_FPS'] = '20'
os.environ['SAMPLE_EVERY_N_FRAMES'] = '20'
os.environ['DELETE_INTERMEDIATE_AFTER_MERGE'] = '1'
os.environ['TRY_LOCAL_DOWNLOAD'] = '1'
os.environ['LOCAL_MIN_FREE_GB'] = '5'
os.environ['LOCAL_SAFETY_MULTIPLIER'] = '1.25'
os.environ['TRIGGER_BROWSER_DOWNLOAD'] = '0'

!python "$SCRIPT_PATH"
